# Matrix Factorization for Sparse Ratings

This notebook solves a small collaborative filtering / matrix completion problem.
The ratings matrix has 20 users and 1000 movies, but only the observed ratings are used for training.


In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/serkanp/ratings-long/ratings_long.csv")
df.head()


,userId,movieId,rating
0,0,16,5
1,0,72,5
2,0,86,5
3,0,259,1
4,0,319,4


In [2]:
num_users = 20
num_movies = 1000

r = np.full((num_users, num_movies), fill_value=np.nan)

for rec in df.itertuples(index=False):
    r[int(rec.userId), int(rec.movieId)] = float(rec.rating)

observed_mask = ~np.isnan(r)

print("rating matrix shape:", r.shape)
print("observed ratings:", observed_mask.sum())
print("missing ratio:", 1 - observed_mask.mean())


rating matrix shape: (20, 1000)
observed ratings: 200
missing ratio: 0.99


The matrix is very sparse: it has 20,000 possible user/movie pairs, but only a small number of ratings are observed.

I model the rating matrix as:

$$\hat R = \mu + U V$$

where:

- $\mu$ is the mean rating in the training set
- $U$ has shape $20 \times 4$
- $V$ has shape $4 \times 1000$

Only the observed entries are included in the loss. Missing values are not treated as zero ratings. Since there are only 200 observed ratings, I also keep a validation split to check whether the model is only memorizing the training ratings.


## Loss Function

For the observed training ratings, I use squared error with L2 regularization. Because the model learns a low-rank correction around the training-set mean, the optimized target is the centered rating $r_{ij} - \mu$:

$$
J(U,V) =
\frac{1}{N} \sum_{(i,j) \in \Omega_{train}} (U_i V_j - (r_{ij} - \mu))^2
+ \frac{\lambda}{N}(||U||_2^2 + ||V||_2^2)
$$

Equivalently, using the final prediction $\hat r_{ij} = \mu + U_i V_j$:

$$
J(U,V) =
\frac{1}{N} \sum_{(i,j) \in \Omega_{train}} (\hat r_{ij} - r_{ij})^2
+ \frac{\lambda}{N}(||U||_2^2 + ||V||_2^2)
$$

Here, $\Omega_{train}$ is the set of observed ratings used for training and $N$ is the number of training ratings. Validation ratings are held out and are not used in the gradient updates.

One important detail is that this matrix factorization objective is **not jointly convex** in $U$ and $V$, because the prediction uses the product $UV$. If one matrix is fixed, the problem is convex in the other matrix, but optimizing both together is non-convex. Gradient descent can still find a useful local solution.


In [3]:
# Split the observed ratings into train and validation sets.
rng = np.random.default_rng(42)

observed_pairs = np.argwhere(observed_mask)
rng.shuffle(observed_pairs)

split_idx = int(0.8 * len(observed_pairs))
train_pairs = observed_pairs[:split_idx]
valid_pairs = observed_pairs[split_idx:]

train_mask = np.zeros_like(observed_mask, dtype=bool)
valid_mask = np.zeros_like(observed_mask, dtype=bool)

train_mask[train_pairs[:, 0], train_pairs[:, 1]] = True
valid_mask[valid_pairs[:, 0], valid_pairs[:, 1]] = True

global_mean = r[train_mask].mean()
r_centered = np.where(train_mask, r - global_mean, 0.0)

print("train ratings:", train_mask.sum())
print("validation ratings:", valid_mask.sum())
print("train mean rating:", round(global_mean, 4))


train ratings: 160
validation ratings: 40
train mean rating: 3.1062


In [4]:
def rmse_on_mask(pred, target, mask):
    clipped_pred = np.clip(pred, 1, 5)
    return np.sqrt(np.mean((clipped_pred[mask] - target[mask]) ** 2))


def fit_matrix_factorization(k, lambda_l2, epochs=5000, learning_rate=0.03, seed=42):
    rng = np.random.default_rng(seed)
    u = rng.normal(loc=0.0, scale=0.1, size=(num_users, k))
    v = rng.normal(loc=0.0, scale=0.1, size=(k, num_movies))

    train_n = train_mask.sum()
    best = {
        "valid_rmse": np.inf,
        "epoch": 0,
        "u": u.copy(),
        "v": v.copy(),
        "train_rmse": np.inf,
    }

    for epoch in range(epochs):
        centered_pred = u @ v

        # Missing entries and validation entries do not affect training gradients.
        error = np.where(train_mask, centered_pred - r_centered, 0.0)

        grad_u = (2 / train_n) * (error @ v.T) + (2 * lambda_l2 / train_n) * u
        grad_v = (2 / train_n) * (u.T @ error) + (2 * lambda_l2 / train_n) * v

        u -= learning_rate * grad_u
        v -= learning_rate * grad_v

        if epoch % 100 == 0 or epoch == epochs - 1:
            pred_now = global_mean + u @ v
            train_rmse = rmse_on_mask(pred_now, r, train_mask)
            valid_rmse = rmse_on_mask(pred_now, r, valid_mask)

            if valid_rmse < best["valid_rmse"]:
                best = {
                    "valid_rmse": valid_rmse,
                    "epoch": epoch,
                    "u": u.copy(),
                    "v": v.copy(),
                    "train_rmse": train_rmse,
                }

    return best


k_values = [4]
lambda_values = [0.1, 0.5, 1.0, 2.0, 5.0]
results = []

for k in k_values:
    for lambda_l2 in lambda_values:
        best_for_setting = fit_matrix_factorization(k=k, lambda_l2=lambda_l2)
        results.append({
            "k": k,
            "lambda_l2": lambda_l2,
            "best_epoch": best_for_setting["epoch"],
            "train_rmse": best_for_setting["train_rmse"],
            "validation_rmse": best_for_setting["valid_rmse"],
            "u": best_for_setting["u"],
            "v": best_for_setting["v"],
        })

results_df = pd.DataFrame([
    {
        "k": row["k"],
        "lambda_l2": row["lambda_l2"],
        "best_epoch": row["best_epoch"],
        "train_rmse": row["train_rmse"],
        "validation_rmse": row["validation_rmse"],
    }
    for row in results
]).sort_values("validation_rmse")

results_df


,k,lambda_l2,best_epoch,train_rmse,validation_rmse
0,4,0.1,4000,0.188891,1.379132
1,4,0.5,4100,0.320517,1.386613
2,4,1.0,4999,0.440699,1.394175
3,4,2.0,4999,0.775160,1.406491
4,4,5.0,0,1.278526,1.413194


In [5]:
best_result = min(results, key=lambda row: row["validation_rmse"])
u = best_result["u"]
v = best_result["v"]
predicted_r = np.clip(global_mean + u @ v, 1, 5)

baseline_pred = np.full_like(r, fill_value=global_mean, dtype=float)
baseline_valid_rmse = rmse_on_mask(baseline_pred, r, valid_mask)

print(f"Best k: {best_result['k']}")
print(f"Best lambda_l2: {best_result['lambda_l2']}")
print(f"Best epoch: {best_result['best_epoch']}")
print(f"Train RMSE: {best_result['train_rmse']:.4f}")
print(f"Validation RMSE: {best_result['validation_rmse']:.4f}")
print(f"Baseline validation RMSE: {baseline_valid_rmse:.4f}")
print("Matrix factorization beats baseline:", best_result["validation_rmse"] < baseline_valid_rmse)
print("u shape:", u.shape)
print("v shape:", v.shape)
print("predicted_r shape:", predicted_r.shape)


Best k: 4
Best lambda_l2: 0.1
Best epoch: 4000
Train RMSE: 0.1889
Validation RMSE: 1.3791
Baseline validation RMSE: 1.4187
Matrix factorization beats baseline: True
u shape: (20, 4)
v shape: (4, 1000)
predicted_r shape: (20, 1000)


## Short Summary

This solution uses low-rank matrix factorization to estimate missing ratings. The model starts from the training-set mean rating and then learns a rank-4 correction with $U$ and $V$.

To avoid training on missing values, the loss is calculated only on observed training ratings. I also split the observed ratings into train and validation sets, so the final RMSE is not only measured on the same ratings used for optimization.

I keep $k=4$ to match the required matrix shapes $U \in \mathbb{R}^{20 \times 4}$ and $V \in \mathbb{R}^{4 \times 1000}$, then compare several L2 regularization values and choose the setting with the lowest validation RMSE. This is better than reporting only training error, because the dataset has only 160 training ratings and can overfit easily.

The optimization problem is not jointly convex in $U$ and $V$, so gradient descent is not guaranteed to find the global optimum. The important part of this experiment is to check validation performance, not just whether the model can fit the observed training ratings.
